# 3D Model Generation from Images

This notebook converts multiple images into a 3D model (GLB format) using:
- **COLMAP**: For Structure-from-Motion and dense reconstruction
- **Open3D**: For mesh processing and reconstruction
- **Trimesh**: For GLB export

## Quick Start

1. **Run Cell 1**: Install dependencies (~2-3 minutes)
2. **Run Cell 2**: Mount Google Drive
3. **Run Cell 3**: Import libraries and setup
4. **Upload Images**: Add 20-50 images to Google Drive folder
5. **Run Last Cell**: Execute the complete pipeline (~20-30 minutes)

## Requirements

- 20-50 images of your object from different angles
- 60-80% overlap between consecutive images
- Good lighting and avoid motion blur

---

## 1. Install Dependencies

Install COLMAP and Python libraries. Takes about 2-3 minutes.

In [ ]:
# Install system dependencies
!sudo apt-get update -qq
!sudo apt-get install -y colmap

# Install Python libraries
!pip install open3d trimesh numpy pillow networkx -q

print("✓ Installation complete!")
print("✓ Using COLMAP + Open3D pipeline")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Import Libraries and Setup

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path
import logging
from datetime import datetime
import numpy as np
import open3d as o3d
import trimesh

# Fix for COLMAP in headless environments (Google Colab)
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['DISPLAY'] = ':0'

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully!")
print(f"✓ Open3D version: {o3d.__version__}")
print("✓ Environment configured for headless execution")

## 4. Configuration

**⚠️ IMPORTANT - Google Colab Limitations:**

Due to Google Colab's headless environment (no display/GPU context), COLMAP **must run on CPU**:
- ✅ `USE_GPU = "0"` - Required for Colab
- ❌ `USE_GPU = "1"` - Will cause OpenGL context errors

**Processing Time:**
- CPU-only mode is slower but reliable
- Expect ~40-60 minutes for 30 images (vs 20-30 with GPU)

Set your paths and parameters below:

In [ ]:
class Config:
    # Google Drive paths
    PROJECT_ROOT = Path("/content/drive/MyDrive/The_Digital_Plate/3D_models")
    INPUT_IMAGES = PROJECT_ROOT / "dish_images/samosa"
    OUTPUT_BASE = PROJECT_ROOT / "dish_models/samosa"
    
    # COLMAP workspace
    COLMAP_WORKSPACE = OUTPUT_BASE / "colmap_workspace"
    COLMAP_DATABASE = COLMAP_WORKSPACE / "database.db"
    COLMAP_IMAGES = COLMAP_WORKSPACE / "images"
    COLMAP_SPARSE = COLMAP_WORKSPACE / "sparse"
    COLMAP_DENSE = COLMAP_WORKSPACE / "dense"
    
    # Final output
    FINAL_OUTPUT = OUTPUT_BASE / "final_model"
    
    # COLMAP settings - DISABLED GPU for Colab compatibility
    CAMERA_MODEL = "SIMPLE_RADIAL"  # Works for most cameras
    MATCHING_METHOD = "exhaustive"   # Best quality matching
    USE_GPU = "0"  # MUST be "0" in Google Colab (no OpenGL context available)
    
    # Open3D mesh settings
    POISSON_DEPTH = 10  # 9-11 recommended (higher = more detail, slower)
    VOXEL_SIZE = 0.002  # For downsampling (smaller = more points)

def setup_directories():
    """Create all required directories"""
    directories = [
        Config.OUTPUT_BASE,
        Config.COLMAP_WORKSPACE,
        Config.COLMAP_IMAGES,
        Config.COLMAP_SPARSE,
        Config.COLMAP_DENSE,
        Config.FINAL_OUTPUT
    ]
    
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)
    
    logger.info("✓ Directories created")
    return True

def verify_setup():
    """Verify Google Drive and check for images"""
    if not Path("/content/drive").exists():
        logger.error("✗ Google Drive not mounted!")
        return False
    
    Config.PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    Config.INPUT_IMAGES.mkdir(parents=True, exist_ok=True)
    
    # Count images
    image_extensions = ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.png', '*.PNG']
    images = []
    for ext in image_extensions:
        images.extend(list(Config.INPUT_IMAGES.glob(ext)))
    
    logger.info(f"✓ Google Drive mounted")
    logger.info(f"✓ Found {len(images)} images in: {Config.INPUT_IMAGES}")
    
    if len(images) == 0:
        logger.warning("⚠ No images found! Please upload images to:")
        logger.warning(f"  {Config.INPUT_IMAGES}")
        return False
    
    return True

# Run setup
verify_setup()
logger.info("⚠ Note: GPU disabled for COLMAP (required for Colab). Processing will be CPU-only.")

## 5. Utility Functions

In [ ]:
def run_command(command, description):
    """Execute a shell command with logging"""
    logger.info(f"{'='*60}")
    logger.info(f"{description}")
    logger.info(f"Command: {' '.join(str(c) for c in command)}")
    logger.info(f"{'='*60}")
    
    try:
        result = subprocess.run(
            command,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        
        if result.stdout:
            logger.info(f"Output: {result.stdout[:500]}...")  # Truncate long output
        
        logger.info(f"✓ {description} - COMPLETED")
        return result
        
    except subprocess.CalledProcessError as e:
        logger.error(f"✗ {description} - FAILED")
        logger.error(f"Error: {e.stderr}")
        raise

def prepare_images():
    """Copy images to COLMAP workspace"""
    logger.info("Preparing images...")
    
    if Config.COLMAP_IMAGES.exists():
        shutil.rmtree(Config.COLMAP_IMAGES)
    Config.COLMAP_IMAGES.mkdir(parents=True)
    
    # Find all images
    image_extensions = ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.png', '*.PNG']
    images = []
    for ext in image_extensions:
        images.extend(list(Config.INPUT_IMAGES.glob(ext)))
    
    if not images:
        raise FileNotFoundError(f"No images found in {Config.INPUT_IMAGES}")
    
    # Copy images
    for img in images:
        shutil.copy2(img, Config.COLMAP_IMAGES / img.name)
    
    logger.info(f"✓ Copied {len(images)} images")
    return len(images)

print("✓ Utility functions defined")

## 6. COLMAP Pipeline

Run Structure-from-Motion and dense reconstruction.

In [ ]:
def colmap_feature_extraction():
    """Extract features from images"""
    command = [
        "colmap", "feature_extractor",
        "--database_path", str(Config.COLMAP_DATABASE),
        "--image_path", str(Config.COLMAP_IMAGES),
        "--ImageReader.camera_model", Config.CAMERA_MODEL,
        "--ImageReader.single_camera", "1",
        "--SiftExtraction.use_gpu", Config.USE_GPU
    ]
    run_command(command, "Feature Extraction")

def colmap_feature_matching():
    """Match features between images"""
    command = [
        "colmap", f"{Config.MATCHING_METHOD}_matcher",
        "--database_path", str(Config.COLMAP_DATABASE),
        "--SiftMatching.use_gpu", Config.USE_GPU
    ]
    run_command(command, "Feature Matching")

def verify_feature_matches():
    """Check if we have enough feature matches for reconstruction"""
    logger.info("Verifying feature matches...")
    
    # Run database stats
    command = [
        "colmap", "database_query",
        "--database_path", str(Config.COLMAP_DATABASE),
        "--statistics"
    ]
    
    try:
        result = subprocess.run(
            command,
            check=False,  # Don't fail if this doesn't work
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        
        if result.stdout:
            logger.info(f"Database statistics:\n{result.stdout}")
        
        # Check for common issues
        if "0 images" in result.stdout or "0 features" in result.stdout:
            logger.error("❌ No features extracted! Check your images.")
            return False
        
        logger.info("✓ Feature matches verified")
        return True
        
    except Exception as e:
        logger.warning(f"Could not verify matches: {e}")
        return True  # Continue anyway

def colmap_sparse_reconstruction():
    """Build sparse 3D model (Structure-from-Motion)"""
    (Config.COLMAP_SPARSE / "0").mkdir(parents=True, exist_ok=True)
    
    command = [
        "colmap", "mapper",
        "--database_path", str(Config.COLMAP_DATABASE),
        "--image_path", str(Config.COLMAP_IMAGES),
        "--output_path", str(Config.COLMAP_SPARSE),
        "--Mapper.ba_refine_focal_length", "1",
        "--Mapper.ba_refine_extra_params", "1",
        "--Mapper.min_num_matches", "15",  # Lower threshold for difficult cases
        "--Mapper.init_min_num_inliers", "100",
        "--Mapper.abs_pose_min_num_inliers", "30",
        "--Mapper.abs_pose_min_inlier_ratio", "0.25",
        "--Mapper.filter_min_tri_angle", "1.5",  # More lenient
        "--Mapper.multiple_models", "0"  # Try to create single model
    ]
    
    try:
        run_command(command, "Sparse Reconstruction")
        
        # Check if model was created
        model_dir = Config.COLMAP_SPARSE / "0"
        required_files = ["cameras.bin", "images.bin", "points3D.bin"]
        
        missing_files = [f for f in required_files if not (model_dir / f).exists()]
        
        if missing_files:
            logger.error(f"❌ Model files missing: {missing_files}")
            logger.error("Sparse reconstruction failed - model not created")
            raise RuntimeError("Failed to create sparse model")
        
        logger.info("✓ Sparse model created successfully")
        
    except subprocess.CalledProcessError as e:
        logger.error("❌ COLMAP mapper failed!")
        logger.error("\nPossible reasons:")
        logger.error("1. ❌ Not enough feature matches between images")
        logger.error("2. ❌ Images don't have sufficient overlap (need 60-80%)")
        logger.error("3. ❌ Images are too different (lighting, angle, etc.)")
        logger.error("4. ❌ Too few images (need at least 10-15 images)")
        logger.error("5. ❌ Images are blurry or low quality")
        logger.error("\n💡 Try:")
        logger.error("  - Add more images with better overlap")
        logger.error("  - Ensure consistent lighting")
        logger.error("  - Use higher resolution images")
        logger.error("  - Take images in a more systematic pattern")
        raise

def colmap_image_undistortion():
    """Undistort images for dense reconstruction"""
    command = [
        "colmap", "image_undistorter",
        "--image_path", str(Config.COLMAP_IMAGES),
        "--input_path", str(Config.COLMAP_SPARSE / "0"),
        "--output_path", str(Config.COLMAP_DENSE),
        "--output_type", "COLMAP"
    ]
    run_command(command, "Image Undistortion")

def colmap_dense_reconstruction():
    """Compute dense depth maps"""
    command = [
        "colmap", "patch_match_stereo",
        "--workspace_path", str(Config.COLMAP_DENSE),
        "--workspace_format", "COLMAP",
        "--PatchMatchStereo.geom_consistency", "true"
    ]
    run_command(command, "Dense Stereo Reconstruction")

def colmap_stereo_fusion():
    """Generate dense point cloud"""
    output_ply = Config.COLMAP_DENSE / "fused.ply"
    
    command = [
        "colmap", "stereo_fusion",
        "--workspace_path", str(Config.COLMAP_DENSE),
        "--workspace_format", "COLMAP",
        "--input_type", "geometric",
        "--output_path", str(output_ply)
    ]
    run_command(command, "Stereo Fusion")
    
    logger.info(f"✓ Point cloud saved: {output_ply}")
    return output_ply

print("✓ COLMAP functions defined")

## 7. Open3D Mesh Processing

Use Open3D to process point cloud and create high-quality mesh.

In [ ]:
def process_point_cloud_with_open3d(input_ply):
    """Process point cloud and generate mesh using Open3D"""
    logger.info("="*60)
    logger.info("Processing point cloud with Open3D")
    logger.info("="*60)
    
    # Check if input file exists
    if not input_ply.exists():
        raise FileNotFoundError(f"Point cloud file not found: {input_ply}")
    
    # Load point cloud
    logger.info(f"Loading point cloud: {input_ply}")
    pcd = o3d.io.read_point_cloud(str(input_ply))
    logger.info(f"✓ Loaded {len(pcd.points)} points")
    
    if len(pcd.points) == 0:
        raise ValueError("Point cloud is empty!")
    
    # Estimate normals
    logger.info("Estimating normals...")
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.01, max_nn=30)
    )
    pcd.orient_normals_consistent_tangent_plane(30)
    logger.info("✓ Normals estimated")
    
    # Statistical outlier removal
    logger.info("Removing outliers...")
    pcd_clean, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    logger.info(f"✓ Cleaned to {len(pcd_clean.points)} points")
    
    if len(pcd_clean.points) < 1000:
        logger.warning("Very few points after cleaning. Results may be poor.")
    
    # Poisson surface reconstruction
    logger.info(f"Creating mesh (Poisson depth={Config.POISSON_DEPTH})...")
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd_clean, 
        depth=Config.POISSON_DEPTH,
        width=0,
        scale=1.1,
        linear_fit=False
    )
    logger.info(f"✓ Mesh created: {len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles")
    
    # Remove low density vertices
    logger.info("Removing low-density vertices...")
    vertices_to_remove = densities < np.quantile(densities, 0.1)
    mesh.remove_vertices_by_mask(vertices_to_remove)
    logger.info(f"✓ Final mesh: {len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles")
    
    # Clean up mesh
    logger.info("Cleaning mesh...")
    mesh.remove_degenerate_triangles()
    mesh.remove_duplicated_triangles()
    mesh.remove_duplicated_vertices()
    mesh.remove_non_manifold_edges()
    logger.info("✓ Mesh cleaned")
    
    # Smooth mesh
    logger.info("Smoothing mesh...")
    mesh = mesh.filter_smooth_simple(number_of_iterations=2)
    mesh.compute_vertex_normals()
    logger.info("✓ Mesh smoothed")
    
    # Save mesh
    output_ply = Config.FINAL_OUTPUT / "mesh_open3d.ply"
    o3d.io.write_triangle_mesh(str(output_ply), mesh)
    logger.info(f"✓ Mesh saved: {output_ply}")
    
    return output_ply

print("✓ Open3D processing functions defined")

## 8. Convert to GLB Format

In [ ]:
def convert_to_glb(input_mesh):
    """Convert mesh to GLB format using trimesh"""
    logger.info("="*60)
    logger.info("Converting to GLB format")
    logger.info("="*60)
    
    try:
        output_glb = Config.FINAL_OUTPUT / "model.glb"
        
        # Load mesh with trimesh
        logger.info(f"Loading mesh: {input_mesh}")
        mesh = trimesh.load(str(input_mesh))
        
        # Export to GLB
        logger.info("Exporting to GLB...")
        mesh.export(str(output_glb), file_type='glb')
        
        if output_glb.exists():
            file_size = output_glb.stat().st_size / 1024 / 1024
            logger.info(f"✓ GLB saved: {output_glb} ({file_size:.2f} MB)")
        else:
            logger.warning("GLB file was not created")
            return input_mesh
        
        return output_glb
        
    except Exception as e:
        logger.error(f"GLB conversion failed: {e}")
        logger.info(f"PLY file available at: {input_mesh}")
        return input_mesh

print("✓ GLB conversion function defined")

## 9. Complete Pipeline

**⚠️ IMPORTANT: Read Before Running**

This cell will:
1. Prepare images
2. Extract and match features (COLMAP)
3. Build sparse 3D model (COLMAP)
4. Generate dense point cloud (COLMAP)
5. Create high-quality mesh (Open3D)
6. Export to GLB format

**How to run:**
1. **Make sure images are uploaded** to Google Drive folder
2. **Uncomment the last line** in this cell (remove the `#`)
3. **Run this cell** (Shift+Enter or click the play button)
4. **Wait ~20-30 minutes** for processing

**Note:** The function is defined but won't run until you uncomment the last line!

In [ ]:
def run_complete_pipeline():
    """Execute the complete 3D reconstruction pipeline"""
    try:
        start_time = datetime.now()
        
        logger.info("\n" + "="*80)
        logger.info("🚀 STARTING 3D RECONSTRUCTION PIPELINE")
        logger.info("Pipeline: COLMAP + Open3D")
        logger.info("="*80 + "\n")
        
        # Verify setup
        logger.info("[1/10] Verifying setup...")
        if not verify_setup():
            raise RuntimeError("Setup verification failed!")
        
        # Prepare images
        logger.info("\n[2/10] Preparing images...")
        num_images = prepare_images()
        logger.info(f"✓ {num_images} images ready\n")
        
        if num_images < 10:
            logger.warning("⚠️  WARNING: Less than 10 images detected!")
            logger.warning("   Sparse reconstruction may fail with too few images.")
            logger.warning("   Recommended: 20-30+ images with good overlap")
        
        # COLMAP: Feature extraction
        logger.info("[3/10] Extracting features...")
        colmap_feature_extraction()
        
        # COLMAP: Feature matching
        logger.info("\n[4/10] Matching features...")
        colmap_feature_matching()
        
        # Verify matches before continuing
        logger.info("\n[4.5/10] Verifying feature matches...")
        verify_feature_matches()
        
        # COLMAP: Sparse reconstruction
        logger.info("\n[5/10] Building sparse 3D model...")
        colmap_sparse_reconstruction()
        
        # COLMAP: Image undistortion
        logger.info("\n[6/10] Undistorting images...")
        colmap_image_undistortion()
        
        # COLMAP: Dense reconstruction
        logger.info("\n[7/10] Computing dense depth maps...")
        colmap_dense_reconstruction()
        
        # COLMAP: Generate point cloud
        logger.info("\n[8/10] Generating dense point cloud...")
        point_cloud = colmap_stereo_fusion()
        
        # Open3D: Process and create mesh
        logger.info("\n[9/10] Creating high-quality mesh with Open3D...")
        mesh_file = process_point_cloud_with_open3d(point_cloud)
        
        # Check if mesh was created
        if not mesh_file.exists():
            raise FileNotFoundError(f"Mesh file was not created: {mesh_file}")
        
        # Convert to GLB
        logger.info("\n[10/10] Converting to GLB format...")
        final_model = convert_to_glb(mesh_file)
        
        # Ensure we have a final model to return
        if not final_model or not Path(final_model).exists():
            logger.warning("GLB conversion failed, returning mesh file")
            final_model = mesh_file
        
        # Summary
        end_time = datetime.now()
        duration = end_time - start_time
        
        logger.info("\n" + "="*80)
        logger.info("🎉 PIPELINE COMPLETED SUCCESSFULLY!")
        logger.info("="*80)
        logger.info(f"⏱️  Processing time: {duration}")
        logger.info(f"📸 Images processed: {num_images}")
        logger.info(f"📁 Output directory: {Config.FINAL_OUTPUT}")
        logger.info(f"✨ Final model: {final_model}")
        
        # List all output files
        output_files = sorted(Config.FINAL_OUTPUT.glob("*"))
        if output_files:
            logger.info("\n📦 Generated files:")
            for f in output_files:
                if f.is_file():
                    size_mb = f.stat().st_size / 1024 / 1024
                    logger.info(f"  ✓ {f.name} ({size_mb:.2f} MB)")
        
        logger.info("\n" + "="*80)
        logger.info("🎯 Your 3D model is ready!")
        logger.info(f"📂 Location: {Config.FINAL_OUTPUT}")
        logger.info("="*80 + "\n")
        
        return final_model
        
    except Exception as e:
        logger.error("\n" + "="*80)
        logger.error("❌ PIPELINE FAILED!")
        logger.error("="*80)
        logger.error(f"Error: {str(e)}")
        
        import traceback
        logger.error("\nFull traceback:")
        logger.error(traceback.format_exc())
        logger.error("="*80)
        raise


In [ ]:
# ========================================
# RUN THE PIPELINE
# ========================================
# 
# TO START PROCESSING:
# 1. Make sure you have uploaded images to Google Drive
# 2. Uncomment the line below (remove the # at the beginning)
# 3. Run this cell
#
# final_model = run_complete_pipeline()

## Tips & Troubleshooting

### Image Capture Tips
- **Number of images**: 30-50 images minimum
- **Overlap**: 60-80% overlap between consecutive shots
- **Coverage**: Capture object from all angles (360° around, top and bottom)
- **Lighting**: Use consistent, diffused lighting
- **Focus**: Ensure images are sharp and in focus
- **Background**: Simple, non-reflective background works best

### Performance Tips
- **GPU**: Enable GPU runtime for 2-3x faster processing
  - Runtime → Change runtime type → GPU
- **Image size**: Resize large images (>4MP) to speed up processing
- **Keep active**: Keep Colab tab open to prevent disconnection

### Configuration
Adjust these in the Configuration cell:
- `POISSON_DEPTH`: Higher = more detail (9-11 recommended)
- `USE_GPU`: Set to "0" if no GPU available
- `CAMERA_MODEL`: Change if using fisheye or other special lens

### Common Issues
1. **"No images found"**: Upload images to the correct Google Drive folder
2. **Out of memory**: Try reducing `POISSON_DEPTH` or use fewer images
3. **Poor reconstruction**: Ensure good image overlap and lighting
4. **GPU errors**: Set `Config.USE_GPU = "0"` in configuration

### Output Files
- `model.glb`: Final 3D model (use in AR, web viewers, etc.)
- `mesh_open3d.ply`: Mesh file (can open in MeshLab, Blender)
- `colmap_workspace/dense/fused.ply`: Raw point cloud from COLMAP

### Typical Processing Times
- 30 images: ~20-25 minutes with GPU
- 50 images: ~30-40 minutes with GPU
- Without GPU: Add 50-100% more time

---

**Made with ❤️ using COLMAP + Open3D**